In [ ]:
import os
import glob
import json
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
import librosa

# Function to parse annotations from JSON files
def parse_json_annotations(json_file_path, audio_duration):
    annotations = []
    try:
        with open(json_file_path, 'r') as file:
            data = json.load(file)
            for entry in data:
                start = entry.get("start", 0)
                end = entry.get("end", 0)
                label = entry.get("label", 0)  # Assuming labels are already 0 (non-apnea) or 1 (apnea)

                # Adjust invalid annotations
                start = max(0, start)
                end = min(audio_duration, end)
                if start < end:
                    annotations.append({"start": start, "end": end, "label": label})
    except Exception as e:
        print(f"Error parsing {json_file_path}: {e}")
    return annotations

# Function to extract statistical features from audio
def extract_features(wav_file_path, annotations, sr=8000):
    features = []
    labels = []
    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        audio_length = len(audio) / sr

        for annotation in annotations:
            start = annotation.get("start", 0)
            end = annotation.get("end", 0)

            start_sample = int(start * sr)
            end_sample = int(end * sr)

            if end_sample > start_sample:
                segment = audio[start_sample:end_sample]

                # Extract statistical features
                zcr = np.mean(librosa.feature.zero_crossing_rate(y=segment))
                rmse = np.mean(librosa.feature.rms(y=segment))
                spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr))
                spectral_bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=segment, sr=sr))
                spectral_flatness = np.mean(librosa.feature.spectral_flatness(y=segment))
                spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=segment, sr=sr))

                # Combine features into a single vector
                feature_vector = [zcr, rmse, spectral_centroid, spectral_bandwidth, spectral_flatness, spectral_rolloff]
                features.append(feature_vector)
                labels.append(annotation.get("label", 0))
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")
    return np.array(features), np.array(labels)

# CNN model definition
def build_1d_cnn(input_shape):
    model = Sequential([
        Conv1D(64, kernel_size=2, activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=1),
        Dropout(0.3),

        Conv1D(128, kernel_size=2, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=1),
        Dropout(0.3),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')  
    ])
    return model


# Training function
def train_1d_cnn_model(wav_folder_path, json_folder_path, sr=8000, batch_size=32, epochs=30):
    all_features = []
    all_labels = []

    # Collect features and labels
    wav_files = glob.glob(os.path.join(wav_folder_path, "*.wav"))
    json_files = glob.glob(os.path.join(json_folder_path, "*.json"))

    for wav_file_path in wav_files:
        wav_base = os.path.splitext(os.path.basename(wav_file_path))[0]
        matching_json_file = next((json for json in json_files if wav_base in os.path.basename(json)), None)
        if not matching_json_file:
            print(f"No matching JSON file for {wav_file_path}")
            continue

        audio, _ = librosa.load(wav_file_path, sr=sr)
        audio_duration = len(audio) / sr

        annotations = parse_json_annotations(matching_json_file, audio_duration)
        features, labels = extract_features(wav_file_path, annotations, sr)
        if features.size > 0:
            all_features.extend(features)
            all_labels.extend(labels)

    if not all_features or not all_labels:
        print("No data available for training.")
        return

    # Convert features and reshape for Conv1D
    all_features = np.array(all_features, dtype=np.float32)
    all_labels = np.array(all_labels, dtype=np.float32)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(all_features, all_labels, test_size=0.2, random_state=42)

    # Compute class weights to handle class imbalance
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = dict(enumerate(class_weights))

    # Build and compile the model
    model = build_1d_cnn((all_features.shape[1], 1))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

    # Reshape for Conv1D
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # Callbacks
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1, restore_best_weights=True)

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")

    # Train the model
    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[lr_scheduler, early_stopping],
        class_weight=class_weights
    )

    # Evaluate the model
    '''
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")
    '''
    results = model.evaluate(X_test, y_test)
    loss, accuracy, precision, recall = results
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}, Test Precision: {precision}, Test Recall: {recall}")


    # Generate predictions and classification report
    y_pred = model.predict(X_test).flatten()
    y_pred_classes = (y_pred > 0.5).astype(int)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes))

# Specify folder paths and execute
wav_folder_path = "/Users/ybys/Desktop/TP/PSG_Audio/APNEA_EDF/test"
json_folder_path = "/Users/ybys/Desktop/TP/PSG_Audio/os_apnea/train"
train_1d_cnn_model(wav_folder_path, json_folder_path)